# LevelGenAI — Colab training

Mirrors `Tools/LevelGenAI/README.md`'s "Training on Google Colab" section step by step.
Run cells top to bottom, in order.

**Before running anything:** `Runtime > Change runtime type > T4 GPU`.

Two Colab-specific things to plan around (not a persistent machine):
- the VM's local disk is wiped every time the runtime disconnects — checkpoints must go to Drive, not the repo's local `checkpoints/`;
- free-tier sessions disconnect on idle (~90 min) or after a hard ~12h cap — if that happens mid-run, skip to the **"Resume after a disconnect"** cell instead of restarting from scratch.

## 1. Clone the repo

`data/catalog.json`, `data/corpus/`, and `data/snapshots/dataset_v1.jsonl` are already committed — nothing needs copying in from the Unity checkout.

In [ ]:
!git clone https://github.com/thinhnv-funtom/smash-market-levelgen-ai.git
%cd smash-market-levelgen-ai
!ls data/catalog.json data/corpus/prod13 data/snapshots/dataset_v1.jsonl

## 2. Check the GPU + torch build

Colab already ships a `torch` matched to its CUDA driver. **Don't** run `pip install -r requirements.txt` unless this print shows `cuda: False` — reinstalling a mismatched torch is how you lose GPU access.

In [ ]:
import torch
print(torch.__version__, "cuda:", torch.cuda.is_available())

## 3. Mount Drive and checkpoint there

`checkpoints/` inside the cloned repo lives on the VM and is gone the moment the runtime resets — checkpoint to Drive instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/LevelGenAI/checkpoints'

## 4. Smoke test

Tiny model, a handful of steps — seconds, not minutes. Confirms shapes/dtypes/loss behave before spending real GPU time. Expect `train_loss`/`val_loss` printed and both trending down; fix any crash here before scaling up.

In [ ]:
!PYTHONPATH=src python -m levelgenai.train --snapshot data/snapshots/dataset_v1.jsonl \
    --checkpoint-dir /tmp/smoke_test \
    --n-layer 2 --n-head 2 --d-model 32 --batch-size 4 --max-steps 20 --eval-interval 10

## 5. Train for real

Model size isn't defaulted for a reason — compute isn't the bottleneck, the ~791-level dataset is (see `model.py`'s docstring). The sizes below are a reasonable starting point; sweep a couple and compare `val_loss` rather than trusting one blindly.

Saves **two** checkpoints on every `--eval-interval`: `last.pt` (always — what `--resume` continues from) and `best.pt` (only on a new best `val_loss` — what `generate.py`/the Unity tool should point at).

Watch for `val_loss` climbing back up while `train_loss` keeps falling (overfitting on a small dataset) — if that happens fast, prefer a smaller model or more `--dropout` over a bigger model. There's no early stopping (`--max-steps` runs to completion); Ctrl-C once `val_loss` has clearly stopped improving — `best.pt` already has the best checkpoint saved, so stopping early loses nothing.

In [ ]:
!PYTHONPATH=src python -m levelgenai.train --snapshot data/snapshots/dataset_v1.jsonl \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --n-layer 6 --n-head 6 --d-model 192 --batch-size 16 --max-steps 20000 --eval-interval 500

## Resume after a disconnect

Only run this if training got cut off (crash / Ctrl-C / Colab disconnect). Re-run cells 1–3 first (clone + mount Drive — `last.pt` is still on Drive even though the VM is fresh), then resume instead of restarting from step 0. Skip this cell entirely on a normal, uninterrupted run.

The saved architecture wins on resume — `--n-layer`/`--n-head`/`--d-model`/`--dropout` are ignored; only `--max-steps` and similar schedule flags still apply.

In [ ]:
!PYTHONPATH=src python -m levelgenai.train --snapshot data/snapshots/dataset_v1.jsonl \
    --checkpoint-dir {CHECKPOINT_DIR} --resume {CHECKPOINT_DIR}/last.pt --max-steps 20000

## 6. Sanity-check generation

Confirms the trained checkpoint produces *something* parseable before wiring it into the Unity tool. Early in training expect a low (or 0) accept rate — that's expected, not a bug.

In [ ]:
!PYTHONPATH=src python -m levelgenai.generate --checkpoint {CHECKPOINT_DIR}/best.pt \
    --snapshot data/snapshots/dataset_v1.jsonl \
    --difficulty 0 --object-count-bucket 2 --move-count 24 --num-samples 8 \
    --output-dir /content/scratch_test
!cat /content/scratch_test/summary.json

## 7. (Optional) Generate a real batch here instead of locally

`generate.py` has no KV-cache — every sampled token re-runs the forward pass over the whole context so far, so a dense level is genuinely slow on a local CPU-only `torch`. Colab's GPU makes this much cheaper — do real candidate generation here, not on whatever machine runs the Unity Editor. Adjust `combos` freely.

In [ ]:
import itertools, json, zipfile
from pathlib import Path

OUT_DIR = "/content/batch_out"
combos = itertools.product([0, 1, 2], [1, 3, 5])  # (difficulty, object_count_bucket) — adjust freely

for difficulty, bucket in combos:
    out = f"{OUT_DIR}/d{difficulty}_b{bucket}"
    !PYTHONPATH=src python -m levelgenai.generate --checkpoint {CHECKPOINT_DIR}/best.pt \
        --snapshot data/snapshots/dataset_v1.jsonl \
        --difficulty {difficulty} --object-count-bucket {bucket} --move-count 24 \
        --num-samples 16 --output-dir {out}
    summary = json.load(open(f"{out}/summary.json"))
    print(f"d{difficulty} b{bucket}: {summary['accepted']}/{summary['requested']} accepted")

with zipfile.ZipFile("/content/batch_out.zip", "w") as zf:
    for path in Path(OUT_DIR).rglob("*.json"):
        zf.write(path, path.relative_to(OUT_DIR))

Download the zip, or save straight to Drive instead if you'd rather not deal with a browser download:

In [ ]:
from google.colab import files
files.download("/content/batch_out.zip")

What you get is the same plain level JSON `generate.py` always produces. To get an accepted one into Unity, you don't need the AI Level Generator window's subprocess round-trip at all — just point `Tools > Smash Market > Level SO Generator` at a folder containing the `sample_*.json` files whose `summary.json` entry says `"accepted": true`.

## 8. Get `best.pt` back to wherever the Unity tool runs

It's already durable on Drive. Either sync/download `best.pt` from Drive into the local checkout's `Tools/LevelGenAI/checkpoints/`, or commit it from here so `git pull` picks it up elsewhere:

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!cp {CHECKPOINT_DIR}/best.pt checkpoints/best.pt
!git add checkpoints/best.pt
!git commit -m "Add trained checkpoint from Colab"

Pushing needs a token — Colab has no stored GitHub credentials. Don't paste one in plaintext into a cell (notebooks get shared/committed); enter it interactively instead. Use a fine-grained Personal Access Token scoped to `contents: write` on just this repo.

In [ ]:
from getpass import getpass
token = getpass("GitHub token: ")
!git push https://{token}@github.com/thinhnv-funtom/smash-market-levelgen-ai.git master